# 24. XGBoost under the target-encoded representation

**One variable against ledger row 17** (`lgbm_bag08_seed42_te`, CV 0.966782): the
same nested target and frequency encoder, the same 36 features, the same five folds,
the same seed and the same `learning_rate * n_estimators = 100` budget convention.
The only thing that changes is the learner.

This is the same shape as `17_catboost_te.ipynb`, which asked the same question about
CatBoost and answered it.

## Why this is being run

`NOTES.md` records XGBoost as **dropped rather than tested**. The `06` amendment
called it "a third histogram GBDT on the same 12 features and there is no reason left
to expect it to behave differently, so it is dropped rather than queued". That
judgement was made on 2026-08-04, on the raw 12 features, in the same pre-encoding
regime that produced the CatBoost rejection which was later overturned.

The file already states the honest status: **untested under the representation that
mattered, not rejected.** This run settles that, in either direction.

## The prior is low, and it is written down before the run

Three reasons to expect little, recorded so that a null result reads as a confirmed
prediction rather than a disappointment.

- **The mechanism that paid for CatBoost does not transfer.** CatBoost's ordered
  target statistics are a different way of handling categoricals; XGBoost has no such
  mechanism, so the specific reason the CatBoost reversal happened is absent here.
- **The row 34 ceiling.** A 0.026 single-model improvement converted to +0.000043 in
  the 24-member stack. A 25th member that is merely comparable has less room than
  that.
- **`num_leaves` just returned a null** under this same representation, ledger rows 35
  to 37. The reopening argument identifies decisions worth revisiting; it does not
  predict that revisiting them pays.

Set against that, the two reopenings that did pay were both argued down in advance on
reasoning this specific, and both were wrong. That is the whole reason this is being
measured rather than argued.

## What this notebook decides, and what it does not

**The single-model CV is description, not the verdict.** Rows 16 and 33 are why: the
neural model was 0.0247 behind and still earned a **+0.1178** coefficient in the
fitted stack, because a combiner can use a weak member as a correction where an
equal-weight blend can only average it in.

So the decision is deferred to the stack, and the rule is pre-registered here:

> XGBoost joins if its leave-one-out contribution to a fold-wise 25-member stack is
> **positive on at least 4 of the 5 folds and at least +0.00005 on the mean**, which
> is the floor `17_catboost_te.ipynb` used for the same decision about CatBoost.

That measurement is a separate notebook and a separate ledger row, because it changes
a different variable (the member set, 24 to 25). It runs locally, where all 24 member
vectors already exist, which is also how rows 25, 27, 32 and 34 were produced.

**No submission csv is written here** for the same reason.

## What runs

1. Data, folds, the leak checklist. Lifted from `23`.
2. The encoder, fingerprinted against `13`. Lifted from `23`, so it cannot drift.
3. The leak checks by execution, reproducing `13`'s printed numbers.
4. Bench and determinism at a short budget, then the projection.
5. Five folds at the full budget, the out-of-fold vector, the paired comparison.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Row 17's budget convention, held: learning_rate * n_estimators = 100. The same
# convention 06 and 17 used for CatBoost, so the three learners are compared at a
# matched budget rather than at their own best settings.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0

# Row 17 ran on Kaggle at n_jobs=-1. Matching its thread setting is part of matching
# it; a better local value would make this a comparison across two thread counts.
N_JOBS = -1

# XGBoost's own tree-structure defaults are kept, exactly as 17 kept CatBoost's. The
# question here is what the learner does out of the box on this representation, and
# tuning one arm of a three-way comparison would answer a different one. max_depth=6
# is XGBoost's default and gives up to 64 leaves, against LightGBM's 31; rows 35 to 37
# measured that range to be a plateau, so the difference is not expected to matter.
MAX_DEPTH = 6

# Ledger row 17: this feature set, these folds, this seed, LightGBM.
BASELINE_NAME = "lgbm_bag08_seed42_te"
BASELINE_CV = 0.966782
EXPECTED_FOLD_SHA = "ec282b0968059676"

# Ledger row 26, the other learner on this identical feature set.
CATBOOST_CV = 0.966915

# 13_target_encoding.ipynb printed these. The encoder here must reproduce them.
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   lr {LR}   n_estimators {N_EST}   max_depth {MAX_DEPTH}")

## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up in NOTES.md: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones in `NOTES.md`. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. XGBoost, bench and determinism

XGBoost gets the identical 36 columns LightGBM got. The three categoricals stay in as
native categoricals through `enable_categorical=True`, which is the counterpart of
LightGBM's native handling and CatBoost's `cat_features`, and their target and
frequency encodings are added alongside. That is duplicative, and it is duplicative
for the other two learners too, which is what makes this one variable rather than two.

Numeric NaN is left to XGBoost's native default-direction handling, which is the
counterpart of LightGBM's native routing. No imputation, per the null result in
ledger row 18.

The determinism check trains the same configuration twice inside this one kernel and
requires bit-identical predictions. Comparing against a number from another notebook
was never a clean test.

In [ ]:
import xgboost as xgb


def make(n_est):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )


def run_fold(fold, n_est, want_test=False):
    """Encode inside the fold, train on the rest, predict the fold."""
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
    m = make(n_est)
    t0 = time.time()
    m.fit(Xtr, y[tr])
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if want_test else None
    return {"p": p, "va": va, "secs": secs, "p_te": p_te,
            "auc": float(roc_auc_score(y[va], p))}


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "24_xgboost_te.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    """Print, and append to a log file flushed on every write.

    nbconvert writes the notebook only once the whole run finishes, so without this
    there is no way to watch a long run from outside the kernel. Carried from 06, 17
    and 23.
    """
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, depth={MAX_DEPTH}, n_jobs={N_JOBS} ===")

a = run_fold(PROBE_FOLD, BENCH_EST)
b = run_fold(PROBE_FOLD, BENCH_EST)
delta = float(np.abs(a["p"] - b["p"]).max())
DETERMINISTIC = delta == 0.0

per_tree = a["secs"] / BENCH_EST
print(f"xgboost {xgb.__version__}, hist, n_jobs={N_JOBS}")
print(f"{BENCH_EST} trees on fold {PROBE_FOLD}: {hhmm(a['secs'])}, "
      f"AUC {a['auc']:.6f}")
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")
print()
print(f"rate {per_tree * 100:.1f}s per 100 trees")
print(f"  stage 4, five folds x {N_EST} trees: {hhmm(per_tree * N_EST * 5)}"
      f"   (plus the encoder and the test prediction)")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"projected {hhmm(per_tree * N_EST * 5)}")

## Stage 4. The full run

Five folds at the full budget, the encoder rebuilt inside each one, the test set
predicted by every fold model and averaged, which is what every other model in this
repo does.

In [ ]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))
scores = []

t0 = time.time()
for f in range(5):
    r = run_fold(f, N_EST, want_test=True)
    oof[r["va"]] = r["p"]
    test_pred += r["p_te"] / 5
    scores.append(r["auc"])
    done = time.time() - t0
    note(f"  fold {f}: {scores[-1]:.6f}  ({hhmm(r['secs'])}), "
         f"elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")

cv, sd = float(np.mean(scores)), float(np.std(scores))
print()
print(f"XGBoost CV {cv:.6f} +/- {sd:.6f}   [{hhmm(time.time() - t0)}]")

# Fold PROBE_FOLD was trained twice in this kernel, once at the bench budget and once
# here. This checks determinism at the full budget for free, on a different quantity.
note(f"full run done, CV {cv:.6f} +/- {sd:.6f}")

### The paired comparison

Paired per fold against ledger row 17 on the identical folds. The right test when two
models are scored on identical folds is the spread of the per-fold difference and how
many folds it wins, not the fold spread, which is common to both and cancels. That is
the rule `NOTES.md` settled in "Fold spread is the wrong yardstick for a paired
comparison".

Both comparisons below are description. The verdict is the stack contribution, which
is a separate notebook and a separate ledger row, and the rule for it was
pre-registered in the header above.

In [ ]:
base = np.load(locate("te_bag42_oof.npy"))[ROW_IDX]
bf = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(5)])
d = np.array(scores) - bf

print(f"row 17 reproduces its ledger number to {bf.mean() - BASELINE_CV:+.2e}")
if SMOKE:
    print("SMOKE: the saved row 17 vector is being scored on subsampled rows under a")
    print("different fold partition, so every number in this block is EXPECTED to be")
    print("wrong and none of it is a check. It becomes one when SMOKE is False, which")
    print("is the only way this notebook runs on Kaggle.")
print()
print(f"{'':22} {'CV':>10}")
print(f"{'row 17, LightGBM':22} {bf.mean():>10.6f}")
print(f"{'row 26, CatBoost':22} {CATBOOST_CV:>10.6f}")
print(f"{'this run, XGBoost':22} {cv:>10.6f}")
print()
print(f"per-fold differences vs row 17: {np.round(d, 6).tolist()}")
print(f"mean {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
      f"wins {(d > 0).sum()}/5 folds")
print()
print("For scale, from the ledger:")
print("  row 26, CatBoost on this encoder  +0.000132, paired sd 0.000080, 5/5")
print("  06, CatBoost on the raw features  -0.001675 (fold 0 only)")
print()
print("A number below row 17 is not a stop. Row 16's neural model was 0.0247 behind")
print("and earns +0.1178 in the fitted stack. The stack notebook decides.")

if not SMOKE:
    ok = (LEAK_OK and CLEAN and ENCODER_MATCH and DETERMINISTIC and ALIGNED)
    print()
    print("READY FOR THE STACK STEP" if ok else
          "BLOCKED: a check above failed, this vector must not be stacked")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
np.save(OUT / f"{pre}xgb_te_oof.npy", oof)
np.save(OUT / f"{pre}xgb_te_test.npy", test_pred)
print(f"wrote {pre}xgb_te_oof.npy, {pre}xgb_te_test.npy")

# No submission csv. Nothing here has cleared the pre-registered gate, and a csv in
# submissions/ is one command away from spending a slot the gate did not authorise.
print()
print("ledger line:")
print(f"  name    xgb_te")
print(f"  cv_mean {cv:.6f}")
print(f"  cv_std  {sd:.6f}")
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}")
print()
print("The stack step is a separate notebook and a separate ledger row, because it")
print("changes the member set rather than the learner.")